References: 
https://chatgpt.com/c/6aeb2c2e-3300-83ec-9ac8-6d61466b7641

https://podaac.github.io/tutorials/external/SWOT_to_kerchunk.html?utm_source=chatgpt.com

https://podaac.github.io/tutorials/external/SWOT_SSH_dashboard.html?utm_source=chatgpt.com


In [ ]:
from pathlib import Path

import earthaccess
from obstore.store import LocalStore
from obspec_utils.registry import ObjectStoreRegistry
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser

# Authenticate
print('loggin in...')
earthaccess.login()

print('searching for granules...')
# Find one SWOT granule
granules = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_Expert_2.0",
    count=1,
)

remote_url = granules[0].data_links(access="external")[0]

# Download the granule
print('downloading...')
files = earthaccess.download(
    granules,
    local_path="/data/SWOT/ls_podaac",
)


loggin in...
searching for granules...
downloading...


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

/opt/venvs/team-py/lib/python3.12/site-packages/zarr/core/dtype/npy/bytes.py:386: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=1)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/opt/venvs/team-py/lib/python3.12/site-packages/zarr/core/dtype/npy/bytes.py:386: UnstableSpecificationWarning: The data type (NullTerminatedBytes(length=1)) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by o

ValueError: Failed to decode fill_value. Unsupported dtype |S1

In [5]:
import h5py
import numpy as np

local_path = Path(files[0]).resolve()

with h5py.File(local_path, "r") as f:
    def inspect(name, obj):
        if isinstance(obj, h5py.Dataset):
            if obj.dtype.kind in ("S", "U", "O"):
                print(
                    name,
                    obj.dtype,
                    obj.shape,
                    repr(obj.fillvalue),
                )

    f.visititems(inspect)

polarization_karin |S1 (9866, 2) np.bytes_(b'*')


In [8]:

local_path = Path(files[0]).resolve()

with h5py.File(local_path, "r") as f:
    string_vars = []

    def collect(name, obj):
        if isinstance(obj, h5py.Dataset):
            if obj.dtype.kind in ("S", "U", "O"):
                string_vars.append(name.split("/")[-1])

    f.visititems(collect)

print("Dropping:", string_vars)


# Configure local object storage
registry = ObjectStoreRegistry({
    "file:///": LocalStore()
})

# Parse HDF5 metadata without loading measurement arrays
with open_virtual_dataset(
    url=local_path.as_uri(),
    registry=registry,
    parser=HDFParser(drop_variables=string_vars),
    loadable_variables=[],
) as vds:

    print(vds)
    print("Reference metadata (bytes):", vds.vz.nbytes)
    print("Number of chunk references:", vds.vz.nrefs)

    # Save a Kerchunk-compatible JSON
    vds.vz.to_kerchunk(
        "/data/SWOT/swot_virtual_local.json",
        format="json",
    )

Dropping: ['polarization_karin']
<xarray.Dataset> Size: 148MB
Dimensions:                                (num_lines: 9866, num_pixels: 69,
                                            num_sides: 2)
Coordinates:
    latitude                               (num_lines, num_pixels) int32 3MB ...
    longitude                              (num_lines, num_pixels) int32 3MB ...
    latitude_nadir                         (num_lines) int32 39kB ManifestArr...
    longitude_nadir                        (num_lines) int32 39kB ManifestArr...
Dimensions without coordinates: num_lines, num_pixels, num_sides
Data variables: (12/97)
    time                                   (num_lines) float64 79kB ManifestA...
    time_tai                               (num_lines) float64 79kB ManifestA...
    ssh_karin                              (num_lines, num_pixels) int32 3MB ...
    ssh_karin_qual                         (num_lines, num_pixels) uint32 3MB ManifestArray<shape=(9866, 69), dtype=uint32, chunks=(98

In [12]:
from urllib.parse import urlsplit
import icechunk as ic

# Existing local NetCDF and PO.DAAC HTTPS URL
local_path = Path(local_path).resolve()
local_url = local_path.as_uri()

# 1. Parse the local HDF5 file
registry = ObjectStoreRegistry({
    "file:///": LocalStore()
})

vds = open_virtual_dataset(
    url=local_url,
    registry=registry,
    parser=HDFParser(drop_variables=string_vars),
    loadable_variables=[],
)

print(vds)

# 2. Rewrite references to the PO.DAAC HTTPS URL
vds = vds.vz.rename_paths(
    lambda path: remote_url
    if path == local_url else path
)

# 3. Configure Icechunk HTTPS virtual chunk access
parsed = urlsplit(remote_url)
http_prefix = f"{parsed.scheme}://{parsed.netloc}/"

config = ic.RepositoryConfig.default()

config.set_virtual_chunk_container(
    ic.VirtualChunkContainer(
        url_prefix=http_prefix,
        store=ic.http_store(opts={}),
    )
)

# 4. Create persistent local Icechunk repository
storage = ic.local_filesystem_storage(
    "/data/SWOT/swot_icechunk"
)

repo = ic.Repository.create(
    storage,
    config,
)

# 5. Write virtual references
session = repo.writable_session("main")

vds.vz.to_icechunk(session.store)

snapshot = session.commit(
    "Add SWOT SSH granule"
)

repo.save_config()

print("Icechunk snapshot:", snapshot)

<xarray.Dataset> Size: 148MB
Dimensions:                                (num_lines: 9866, num_pixels: 69,
                                            num_sides: 2)
Coordinates:
    latitude                               (num_lines, num_pixels) int32 3MB ...
    longitude                              (num_lines, num_pixels) int32 3MB ...
    latitude_nadir                         (num_lines) int32 39kB ManifestArr...
    longitude_nadir                        (num_lines) int32 39kB ManifestArr...
Dimensions without coordinates: num_lines, num_pixels, num_sides
Data variables: (12/97)
    time                                   (num_lines) float64 79kB ManifestA...
    time_tai                               (num_lines) float64 79kB ManifestA...
    ssh_karin                              (num_lines, num_pixels) int32 3MB ...
    ssh_karin_qual                         (num_lines, num_pixels) uint32 3MB ManifestArray<shape=(9866, 69), dtype=uint32, chunks=(9866, 69)...
    ssh_karin_uncert  

  2026-11-08T22:34:19.102996Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-08T22:34:19.108750Z  WARN icechunk_storage::readback: conditional PUT is enabled but `unsafe_use_metadata` is disabled — lost-response recovery for conditional writes requires user metadata to stamp write-ids; without it, transient PUT failures may surface as spurious conflicts even when the write actually landed. See icechunk_storage::Settings::unsafe_use_metadata.
    at icechunk-storage/src/readback.rs:31



Icechunk snapshot: 38THTPYHMSJ05C067V6G


In [13]:
# Run this as stand-alone
import icechunk as ic
import xarray as xr
from urllib.parse import urlsplit
import earthaccess


# Authenticate with NASA Earthdata
earthaccess.login()

token = earthaccess.get_edl_token()

config = ic.RepositoryConfig.default()

config.set_virtual_chunk_container(
    ic.VirtualChunkContainer(
        url_prefix=http_prefix,
        store=ic.http_store(
            headers={
                "Authorization": f"Bearer {token}",
            }
        ),
    )
)

# Find a SWOT granule
granules = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_Expert_2.0",
    count=1,
)

remote_url = granules[0].data_links(access="external")[0]
print("Remote URL:", remote_url)

# Same HTTPS URL used when creating the Icechunk repository
parsed = urlsplit(remote_url)
http_prefix = f"{parsed.scheme}://{parsed.netloc}/"

storage = ic.local_filesystem_storage(
    "/data/SWOT/swot_icechunk"
)

repo = ic.Repository.open(
    storage,
    authorize_virtual_chunk_access={
        http_prefix: ic.credentials.HttpAccess
    },
)

session = repo.readonly_session("main")


INFO:earthaccess.api:Granules found: 19996


Remote URL: https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_2.0/SWOT_L2_LR_SSH_Expert_001_149_20230726T122755_20230726T131924_PGC0_01.nc


  2026-11-08T22:51:56.270079Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



In [14]:

ds = xr.open_zarr(
    session.store,
    zarr_format=3,
    consolidated=False,
    chunks={},
    decode_cf=False,
    create_default_indexes=False,
)


print(ds)

<xarray.Dataset> Size: 148MB
Dimensions:                                (num_lines: 9866, num_pixels: 69,
                                            num_sides: 2)
Dimensions without coordinates: num_lines, num_pixels, num_sides
Data variables: (12/101)
    ancillary_surface_classification_flag  (num_lines, num_pixels) uint8 681kB dask.array<chunksize=(9866, 69), meta=np.ndarray>
    geoid                                  (num_lines, num_pixels) int32 3MB dask.array<chunksize=(9866, 69), meta=np.ndarray>
    distance_to_coast                      (num_lines, num_pixels) int16 1MB dask.array<chunksize=(9866, 69), meta=np.ndarray>
    cross_track_distance                   (num_lines, num_pixels) float32 3MB dask.array<chunksize=(9866, 69), meta=np.ndarray>
    dynamic_ice_flag                       (num_lines, num_pixels) uint8 681kB dask.array<chunksize=(9866, 69), meta=np.ndarray>
    doppler_centroid                       (num_lines, num_pixels) int16 1MB dask.array<chunksize=(9866, 

In [15]:
ssh = ds["ssh_karin_2"]

subset = ssh.isel({
    dim: slice(0, min(10, size))
    for dim, size in ssh.sizes.items()
}).compute()

print(subset)

StorageError: error fetching virtual reference: The operation lacked valid authentication credentials for path podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_2.0/SWOT_L2_LR_SSH_Expert_001_149_20230726T122755_20230726T131924_PGC0_01.nc: Error performing GET https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_2.0/SWOT_L2_LR_SSH_Expert_001_149_20230726T122755_20230726T131924_PGC0_01.nc in 2.727913662s - Server returned non-2xx status code: 401 Unauthorized: HTTP Basic: Access denied.


context:
   0: icechunk::store::get
           with key="ssh_karin_2/c/0/0" byte_range=From(0)
             at icechunk/src/store.rs:183

In [12]:
import inspect

print("Icechunk version:", ic.__version__)
print(inspect.signature(ic.http_store))
print(inspect.signature(ic.Repository.open))


Icechunk version: 2.3.1
(opts: dict[str, str] | None = None, headers: dict[str, str] | None = None) -> ObjectStoreConfig.Http
(storage: Storage, config: RepositoryConfig | None = None, authorize_virtual_chunk_access: dict[str, Credentials.S3 | Credentials.Gcs | Credentials.Azure | Credentials.LocalFileSystemAccess | Credentials.HttpAccess | None] | None = None, attribution: Attribution | None = None) -> Self


---

In [1]:
import s3fs
import glob
import dask
import ujson
import cartopy
import requests
import earthaccess

In [2]:
earthaccess.login()

granules = earthaccess.search_data(
    short_name="SWOT_L2_LR_SSH_Expert_2.0",
    count=1,
)

url = granules[0].data_links(access="external")[0]

session = earthaccess.get_requests_https_session()

In [3]:
response = session.get(
    url,
    headers={"Range": "bytes=0-1023"},
    timeout=60,
)

print(response.status_code)
print(response.headers.get("Content-Range"))
print(len(response.content))

206
bytes 0-1023/34164922
1024


In [4]:
url = granules[0].data_links(access="external")[0]
print(url)

https://archive.swot.podaac.earthdata.nasa.gov/podaac-swot-ops-cumulus-protected/SWOT_L2_LR_SSH_2.0/SWOT_L2_LR_SSH_Expert_001_149_20230726T122755_20230726T131924_PGC0_01.nc


In [5]:
fs = earthaccess.get_fsspec_https_session()

with fs.open(
    url,
    "rb",
    block_size=1024 * 1024,
    cache_type="none",
) as f:
    print(f.read(8))

b'\x89HDF\r\n\x1a\n'


In [7]:
from kerchunk.hdf import SingleHdf5ToZarr
import json

local_files = earthaccess.download(granules[:1], local_path="../DATA")

with open(local_files[0], "rb") as f:
    refs = SingleHdf5ToZarr(
        f,
        url=url,  # Keep references pointing to PO.DAAC HTTPS
        inline_threshold=0,
    ).translate()

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import fsspec
import xarray as xr

token = earthaccess.get_edl_token()

remote_options = {
    "headers": {
        "Authorization": f"Bearer {token}",
    },
}

ref_fs = fsspec.filesystem(
    "reference",
    fo="swot_single.json",
    remote_protocol="https",
    remote_options=remote_options,
)

mapper = ref_fs.get_mapper("")

ds = xr.open_dataset(
    mapper,
    engine="zarr",
    consolidated=False,
    chunks={},
)

print(ds)

In [ ]:
print(ds.data_vars)

ssh = ds["ssh_karin_2"]

print(ssh.dims)
print(ssh.shape)

subset = ssh.isel(
    {dim: slice(0, min(10, size))
     for dim, size in ssh.sizes.items()}
).compute()

print(subset.values)